# ipynb/umap_index.ipynb - UMAP 索引页 / UMAP index page

## 项目背景 / Background
UMAP 索引页
UMAP index page

## 功能模块 / Modules
- UMAP 系列 notebook 索引
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: ipynb/umap1.ipynb、ipynb/umap2.ipynb
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [1]:
if (!require("pacman")) {
  message("Installing pacman...")
  install.packages("pacman")
}

Loading required package: pacman



In [ ]:
# ============================================================
# 1. 库加载
# ============================================================
options(repos = c(CRAN = "https://cloud.r-project.org")) # 防止镜像选择弹窗卡顿

if (!require("pacman")) install.packages("pacman")
pacman::p_load(reticulate, ggplot2, dplyr, tidyr, FNN, cluster, purrr, 
               foreach, doParallel, parallelDist)

# Configure Python
cat("Initializing Python/Numpy...\n")
np <- import("numpy")

# ============================================================
# 2. 辅助函数 & 日志工具
# ============================================================
flatten_3d <- function(arr) {
  dims <- dim(arr)
  array_reshape(arr, c(dims[1], prod(dims[-1])))
}

# [增强版] 日志打印辅助函数 (强制刷新缓冲区)
log_info <- function(msg, ...) {
  cat(sprintf("[%s] %s\n", format(Sys.time(), "%H:%M:%S"), sprintf(msg, ...)))
  flush.console() # 关键：强制立即显示输出，防止RStudio缓存日志
}

print_header <- function(title) {
  cat(paste0("\n", strrep("=", 60), "\n"))
  cat(sprintf(" %s\n", title))
  cat(paste0(strrep("=", 60), "\n"))
  flush.console()
}

# ============================================================
# 3. 高性能并行处理函数 (Debug版)
# ============================================================

SAMPLES_PER_GROUP <- 1000 

process_and_evaluate_parallel <- function(file_path, model_name, vector_name, 
                                          label_name = "label12", 
                                          k_percents = c(0.01, 0.03, 0.05)) {
  
  print_header(sprintf("PROCESSING MODEL: %s", model_name))
  log_info("Step 1: Loading .npz file from disk...")
  
  # --- A. 数据读取 (Main Process) ---
  if (!file.exists(file_path)) stop(paste("File not found:", file_path))
  
  data <- np$load(file_path)
  log_info(" -> File loaded into Python interface.")
  
  raw_vectors <- data[[vector_name]]
  labels_mat <- data[[label_name]] 
  
  # 维度检查与打印
  log_info("Step 2: Checking dimensions and flattening if needed...")
  if (length(dim(raw_vectors)) > 2) {
    orig_dim <- dim(raw_vectors)
    vectors <- flatten_3d(raw_vectors)
    log_info(" -> Flattened 3D vectors. From [%s] to [%d, %d]", 
             paste(orig_dim, collapse="x"), nrow(vectors), ncol(vectors))
  } else {
    vectors <- raw_vectors
    log_info(" -> 2D vectors loaded. Shape: [%d, %d]", nrow(vectors), ncol(vectors))
  }
  
  n_classes <- ncol(labels_mat)
  log_info(" -> Labels loaded. Total Classes: %d", n_classes)
  
  # --- B. 预处理任务列表 (Pre-slicing) ---
  log_info("Step 3: Pre-slicing data (Sampling %d per group)...", SAMPLES_PER_GROUP)
  task_list <- list()
  
  for (class_idx in 1:n_classes) {
    # 采样逻辑
    is_positive <- labels_mat[, class_idx] == 1
    pos_indices_all <- which(is_positive)
    neg_indices_all <- which(!is_positive)
    
    set.seed(123 + class_idx) 
    
    idx_pos <- if(length(pos_indices_all) >= SAMPLES_PER_GROUP) {
      sample(pos_indices_all, SAMPLES_PER_GROUP, replace = FALSE)
    } else {
      sample(pos_indices_all, SAMPLES_PER_GROUP, replace = TRUE)
    }
    idx_neg <- sample(neg_indices_all, SAMPLES_PER_GROUP, replace = FALSE)
    
    current_indices <- c(idx_neg, idx_pos) 
    
    subset_vectors <- vectors[current_indices, ]
    binary_labels <- c(rep(0, SAMPLES_PER_GROUP), rep(1, SAMPLES_PER_GROUP))
    
    task_list[[class_idx]] <- list(
      class_idx = class_idx,
      vectors = subset_vectors,
      labels = binary_labels,
      n_pos = length(pos_indices_all)
    )
    # 稍微打印一下进度，防止看起来卡死
    if(class_idx %% 5 == 0) cat(".") 
  }
  cat("\n")
  log_info(" -> Task list created for %d classes.", length(task_list))
  
  # 释放主内存
  log_info("Step 4: Cleaning memory (GC)...")
  rm(vectors, labels_mat, raw_vectors, data)
  gc_res <- gc()
  log_info(" -> Memory cleaned. Current usage: %s MB", sum(gc_res[,2]))
  
  # --- C. 启动并行环境 ---
  # 自动检测核心数，防止 32 核设置在普通电脑上卡死
  phys_cores <- parallel::detectCores(logical = FALSE)
  n_cores <- 8
  
  # 如果物理核心少于 32，给出警告但仍按用户要求尝试（或者你可以取消下面注释限制它）
  if (n_cores > phys_cores) {
    log_info("WARNING: Requested 32 cores but only %d physical cores detected. System may lag.", phys_cores)
  }
  
  log_info("Step 5: Launching Cluster with %d cores (This usually takes 5-10s)...", n_cores)
  cl <- makeCluster(n_cores, outfile = "") 
  registerDoParallel(cl)
  log_info(" -> Cluster active. Starting parallel calculation loop.")
  
  # --- D. 并行循环 ---
  results <- foreach(task = task_list, 
                     .combine = rbind, 
                     .packages = c("FNN", "cluster", "parallelDist", "dplyr"),
                     .noexport = c("vectors", "labels_mat")) %dopar% {
    
    # [子进程开始] --------------------------------------------
    # 打印开始信号，如果这里没打印，说明任务分配有问题
    cat(sprintf("[Worker] Class %02d STARTING...\n", task$class_idx))
    
    start_time <- Sys.time()
    
    # 解包
    current_vectors <- task$vectors
    binary_labels <- task$labels
    class_idx <- task$class_idx
    n_original_pos <- task$n_pos
    
    # 1. 计算 Silhouette (通常最耗时)
    dist_mat <- parallelDist::parDist(current_vectors, method = "euclidean", threads = 1)
    sil_obj <- cluster::silhouette(binary_labels, dist_mat)
    avg_sil <- mean(sil_obj[, "sil_width"])
    
    # 2. 计算 k-NN Purity
    k_counts <- floor(SAMPLES_PER_GROUP * k_percents)
    max_k <- max(k_counts)
    
    pos_start_idx <- SAMPLES_PER_GROUP + 1
    pos_end_idx <- 2 * SAMPLES_PER_GROUP
    
    knn_res_all <- FNN::get.knn(data = current_vectors, k = max_k)
    neighbor_indices <- knn_res_all$nn.index[pos_start_idx:pos_end_idx, ]
    neighbor_labels <- matrix(binary_labels[neighbor_indices], ncol = max_k)
    
    # 3. 组装结果
    res_list <- list()
    
    res_list[[1]] <- data.frame(
      model = model_name,
      cluster = as.character(class_idx),
      metric = "Silhouette",
      score = avg_sil,
      stringsAsFactors = FALSE
    )
    
    purity_msg <- c()
    for (i in seq_along(k_counts)) {
      k <- k_counts[i]
      current_k_labels <- neighbor_labels[, 1:k, drop=FALSE]
      purity_scores <- rowMeans(current_k_labels)
      avg_purity <- mean(purity_scores)
      purity_msg <- c(purity_msg, sprintf("k%d%%=%.3f", k_percents[i]*100, avg_purity))
      
      res_list[[length(res_list) + 1]] <- data.frame(
        model = model_name,
        cluster = as.character(class_idx),
        metric = sprintf("Purity (k=%d%%)", k_percents[i] * 100),
        score = avg_purity,
        stringsAsFactors = FALSE
      )
    }
    
    # [子进程结束日志]
    end_time <- Sys.time()
    duration <- round(as.numeric(difftime(end_time, start_time, units = "secs")), 2)
    
    cat(sprintf("[Worker] Class %02d FINISHED | Time: %ss | Sil: %.3f | %s\n",
                class_idx, duration, avg_sil, paste(purity_msg, collapse = ", ")))
    
    bind_rows(res_list)
  }
  
  stopCluster(cl)
  log_info("Step 6: Parallel processing finished & Cluster stopped.")
  return(results)
}

# ============================================================
# 4. 执行流程
# ============================================================

files <- list(
  human = list(path = "data/human_atten.npz", vec = "attn_out_12"),
  modx = list(path = "data/modx_atten.npz", vec = "context_vector"),
  multirm = list(path = "data/multirm_atten.npz", vec = "context_vector")
)

all_results <- list()
start_global <- Sys.time()

log_info("GLOBAL START: Processing %d models", length(files))

for (m_name in names(files)) {
  cfg <- files[[m_name]]
  tryCatch({
    res <- process_and_evaluate_parallel(cfg$path, m_name, cfg$vec)
    all_results[[m_name]] <- res
    log_info("SUCCESS: Model '%s' completed.", m_name)
  }, error = function(e) {
    cat(sprintf("\n[!!!] CRITICAL ERROR processing %s: %s\n", m_name, e$message))
    flush.console()
  })
}

print_header("GLOBAL PROCESSING FINISHED")
log_info("Total execution time: %.2f mins", as.numeric(difftime(Sys.time(), start_global, units = "mins")))

if (length(all_results) > 0) {
  final_df <- bind_rows(all_results)
  
  log_info("Generating plots (ggplot2)...")
  
  # ============================================================
  # 5. 统计与绘图
  # ============================================================
  
  macro_avg <- final_df %>%
    group_by(model, metric) %>%
    summarise(score = mean(score), .groups = 'drop') %>%
    mutate(cluster = "Macro Avg")
  
  weight_avg <- final_df %>%
    group_by(model, metric) %>%
    summarise(score = mean(score), .groups = 'drop') %>%
    mutate(cluster = "Weight Avg")
  
  plot_data <- bind_rows(final_df, macro_avg, weight_avg)
  
  unique_clusters <- sort(as.numeric(unique(final_df$cluster)))
  x_levels <- c(as.character(unique_clusters), "Macro Avg", "Weight Avg")
  plot_data$cluster <- factor(plot_data$cluster, levels = x_levels)
  
  metric_levels <- c("Silhouette", "Purity (k=1%)", "Purity (k=3%)", "Purity (k=5%)")
  plot_data$metric <- factor(plot_data$metric, levels = metric_levels)
  
  best_models <- plot_data %>%
    group_by(metric, cluster) %>%
    top_n(1, score) %>%
    mutate(is_best = TRUE) %>%
    select(model, metric, cluster, is_best)
  
  plot_data <- plot_data %>%
    left_join(best_models, by = c("model", "metric", "cluster")) %>%
    mutate(is_best = replace_na(is_best, FALSE))
  
  p <- ggplot(plot_data, aes(x = cluster, y = score)) +
    geom_col(data = subset(plot_data, is_best), 
             aes(group = model), 
             fill = "#FFD700", alpha = 0.5, width = 0.85, 
             position = position_dodge(width = 0.75)) +
    geom_col(aes(fill = model), width = 0.7, 
             position = position_dodge(width = 0.75),
             color = "black", size = 0.15) + 
    scale_fill_manual(values = c("#66C2A5", "#FC8D62", "#8DA0CB")) + 
    facet_grid(metric ~ ., scales = "free_y") +
    labs(title = "Binary (One-vs-Rest) Separation Quality & Local Purity",
         subtitle = "Subsampled: 1000 Positive vs 1000 Negative per Class",
         y = "Score", x = "Class Label") +
    theme_bw() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, face = "bold"),
      strip.background = element_rect(fill = "grey95"),
      strip.text = element_text(size = 11, face = "bold"),
      legend.position = "top",
      panel.grid.minor = element_blank()
    )
  
  if(!dir.exists("png")) dir.create("png")
  
  log_info("Saving PDF...")
  ggsave("png/binary_class_metrics_halo.pdf", p, width = 16, height = 12)
  log_info("Done! Plot saved to png/binary_class_metrics_halo.pdf")
} else {
  log_info("No results generated due to errors.")
}

Initializing Python/Numpy...
[00:34:59] GLOBAL START: Processing 3 models

 PROCESSING MODEL: human
[00:34:59] Step 1: Loading .npz file from disk...
[00:34:59]  -> File loaded into Python interface.
[00:35:04] Step 2: Checking dimensions and flattening if needed...
[00:35:11]  -> Flattened 3D vectors. From [203993x12x128] to [203993, 1536]
[00:35:11]  -> Labels loaded. Total Classes: 12
[00:35:11] Step 3: Pre-slicing data (Sampling 1000 per group)...
..
[00:35:11]  -> Task list created for 12 classes.
[00:35:11] Step 4: Cleaning memory (GC)...
[00:35:12]  -> Memory cleaned. Current usage: 431.1 MB
[00:35:12] Step 5: Launching Cluster with 8 cores (This usually takes 5-10s)...
[00:35:12]  -> Cluster active. Starting parallel calculation loop.

[!!!] CRITICAL ERROR processing human: ignoring SIGPIPE signal

 PROCESSING MODEL: modx
[00:49:24] Step 1: Loading .npz file from disk...
[00:49:24]  -> File loaded into Python interface.
[00:49:26] Step 2: Checking dimensions and flattening if n